# Проверка рыночного датасета BTCUSDT — 0002

Проверка сохранённой версии без сети, обучения и торговых расчётов. Нужна локальная база `data/datasets/btcusdt_1h_4h_v1/market.sqlite3`; в Git её нет. Дата наблюдений — декабрь 2021 — декабрь 2025, дата сборки — 20 октября 2026. Количества финального периода показывают только техническое покрытие, а не качество прогнозов.


In [1]:
import json
import sqlite3
import sys
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "scripts/market_data.py").exists():
    root = root.parent
if not (root / "scripts/market_data.py").exists():
    raise FileNotFoundError("Откройте ноутбук из корня проекта или notebooks/")
sys.path.insert(0, str(root))
from scripts.market_data import load_config
from src.vkr_data.build import verify_build

summary = json.loads((root / "experiments/0002_market_dataset_summary.json").read_text())
config = load_config(root / "configs/btcusdt_1h_4h.json")
verified = verify_build(config)
assert verified["verified"]
assert verified["snapshot_id"] == summary["build"]["snapshot_id"]
assert verified["dataset_sha256"] == summary["build"]["dataset_sha256"]
print({"verified": True, "rows": verified["rows"], "excluded_rows": verified["excluded_rows"],
       "feature_count": len(summary["build"]["feature_columns"]), "snapshot_id": verified["snapshot_id"]})


{'verified': True, 'rows': 33795, 'excluded_rows': 1269, 'feature_count': 18, 'snapshot_id': '62c6bb707f04c10cf16d9e36671be011537973e286981bbf0ec8612c7d2b5828'}


## Части и причины исключений

При нескольких проблемах сохраняется одна основная причина: граница части, затем часовые входы, затем точные минутные цены. Поэтому количества причин не следует складывать как независимые частоты исходных дефектов.


In [2]:
database = root / summary["database"]
with sqlite3.connect(database.as_uri() + "?mode=ro", uri=True) as connection:
    print("Включённые строки:", connection.execute(
        "SELECT split,count(*) FROM dataset_rows GROUP BY split ORDER BY split").fetchall())
    print("Исключённые строки:", connection.execute(
        "SELECT split,reason,count(*) FROM dataset_exclusions GROUP BY split,reason ORDER BY split,reason").fetchall())
    wrong_timing = connection.execute(
        "SELECT count(*) FROM dataset_rows WHERE signal_time != decision_time+60000 "
        "OR execution_time != decision_time+120000 OR label_end != execution_time+14400000 "
        "OR decision_time%3600000 != 0").fetchone()[0]
    assert wrong_timing == 0
    for split in config["splits"]:
        from datetime import datetime
        left = int(datetime.fromisoformat(split["start"].replace("Z", "+00:00")).timestamp()*1000)
        right = int(datetime.fromisoformat(split["end"].replace("Z", "+00:00")).timestamp()*1000)
        violations = connection.execute(
            "SELECT count(*) FROM dataset_rows WHERE split=? AND "
            "(execution_time<? OR execution_time>=? OR label_end>=? OR label_end<execution_time)",
            (split["name"], left, right, right)).fetchone()[0]
        assert violations == 0, split["name"]
print("Тайминг и полуоткрытые временные границы: OK")
print("Сохранённый результат повторной сборки:", summary["rebuild"]["identical_report"])


Включённые строки: [('final', 8756), ('train', 16259), ('validation', 8780)]
Исключённые строки: [('final', 'split_boundary', 4), ('train', 'invalid_hour', 535), ('train', 'missing_hour', 721), ('train', 'missing_minute', 1), ('train', 'split_boundary', 4), ('validation', 'split_boundary', 4)]
Тайминг и полуоткрытые временные границы: OK
Сохранённый результат повторной сборки: True


## Исходные ответы и ограничения

Для полной сверки с ответами API выполните в корне проекта `python3 scripts/market_data.py validate --config configs/btcusdt_1h_4h.json`. Эта команда проверяет gzip/SHA-256, все импортированные поля SQLite и карантинные строки; к сети она не обращается. `verify_build` выше проверяет содержательные хеши исходной таблицы и результата, но не заменяет сверку с raw.

Ончейн-фичей в этой версии нет: историческое время доступности версий не подтверждено. Рыночная задержка доступности — сценарное допущение. Отсутствующая будущая цена исключает обучающий ответ; она не должна молча превращаться в правило отмены сделки будущего бэктеста.
